# 🖼️ 멀티모달 검색 v2 — 벡터 공간을 활용한 3가지 검색 전략

## 핵심 아이디어

`image_vector`(1024D)와 `mm_vision_text_vector`(1024D)는 **동일한 Azure AI Vision Florence 임베딩 공간**에 있습니다.  
이를 활용하면 이미지↔텍스트 크로스모달 검색이 가능합니다.

## 벡터 필드 구조

| 필드 | 차원 | 임베딩 모델 | 용도 |
|------|------|------------|------|
| `image_vector` | 1024D | AI Vision Florence | 이미지 픽셀 직접 임베딩 |
| `mm_vision_text_vector` | 1024D | AI Vision Florence | 텍스트를 **같은 공간**에 임베딩 |
| `content_vector` | 3072D | text-embedding-3-large | 텍스트 의미 임베딩 |
| `structured_features` | Complex | GPT-5.2 | 9개 구조화 피처 (매칭 근거) |

## 3가지 검색 모드

| 모드 | 쿼리 벡터화 | 검색 대상 필드 | 근거 |
|------|------------|---------------|------|
| **1. 이미지만** | 이미지→`vectorizeImage`→1024D | `image_vector` | `structured_features` |
| **2. 이미지+텍스트** | 이미지→1024D + 텍스트→`vectorizeText`→1024D | `image_vector` + `mm_vision_text_vector` + BM25 | `structured_features` |
| **3. 텍스트만** | 텍스트→`vectorizeText`→1024D + 텍스트→embedding→3072D | `mm_vision_text_vector` + `content_vector` + BM25 | `structured_features` |

---

## 1️⃣ 환경 설정

In [ ]:
import os
import sys
import time
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from IPython.display import display, Image as IPImage

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))
from prompts import IMAGE_ANALYSIS_SYSTEM_PROMPT, IMAGE_ANALYSIS_USER_PROMPT_SEARCH

load_dotenv(override=True)

# Azure AI Search
search_client = SearchClient(
    endpoint=os.getenv("SEARCH_ENDPOINT"),
    index_name=os.getenv("SEARCH_INDEX_NAME"),
    credential=AzureKeyCredential(os.getenv("SEARCH_ADMIN_KEY"))
)

# Azure OpenAI (텍스트 임베딩 3072D + GPT Vision)
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPEN_AI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPEN_AI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION")
)
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# Azure AI Vision (이미지/텍스트 → 1024D Florence)
VISION_ENDPOINT = os.getenv("AZURE_AI_VISION_ENDPOINT").rstrip("/")
VISION_KEY = os.getenv("AZURE_AI_VISION_KEY")

print("✅ 환경 설정 완료")
print(f"   Index: {os.getenv('SEARCH_INDEX_NAME')}")
print(f"   Vision: {VISION_ENDPOINT}")
print(f"   Chat: {CHAT_DEPLOYMENT}")
print(f"   Embedding: {EMBEDDING_DEPLOYMENT}")

✅ 환경 설정 완료
   Index: products-index
   Vision: https://computer-vision-changju-v2.cognitiveservices.azure.com
   Chat: gpt-5.2
   Embedding: text-embedding-3-large


---
## 2️⃣ 벡터화 함수 & 결과 출력

3가지 벡터화 경로:

| 함수 | 입력 | 출력 | 대상 필드 |
|------|------|------|----------|
| `vectorize_image` | 이미지 URL | 1024D | `image_vector` |
| `vectorize_text_vision` | 텍스트 | 1024D | `mm_vision_text_vector` |
| `get_embedding` | 텍스트 | 3072D | `content_vector` |

In [9]:
def vectorize_image(image_url):
    """이미지 URL → 1024D (Azure AI Vision Florence)"""
    api_url = f"{VISION_ENDPOINT}/computervision/retrieval:vectorizeImage?api-version=2024-02-01&model-version=2023-04-15"
    headers = {"Ocp-Apim-Subscription-Key": VISION_KEY, "Content-Type": "application/json"}
    resp = requests.post(api_url, headers=headers, json={"url": image_url}, timeout=30)
    if not resp.ok:
        print(f"   ⚠️ 이미지 벡터화 실패 (HTTP {resp.status_code}): {resp.text[:100]}")
        return None
    return resp.json()["vector"]


def vectorize_text_vision(text):
    """텍스트 → 1024D (Azure AI Vision Florence, image_vector와 같은 공간)"""
    api_url = f"{VISION_ENDPOINT}/computervision/retrieval:vectorizeText?api-version=2024-02-01&model-version=2023-04-15"
    headers = {"Ocp-Apim-Subscription-Key": VISION_KEY, "Content-Type": "application/json"}
    resp = requests.post(api_url, headers=headers, json={"text": text}, timeout=30)
    resp.raise_for_status()
    return resp.json()["vector"]


def get_embedding(text):
    """텍스트 → 3072D (text-embedding-3-large)"""
    response = openai_client.embeddings.create(input=text, model=EMBEDDING_DEPLOYMENT)
    return response.data[0].embedding


def auto_analyze_image(image_url):
    """GPT Vision으로 이미지를 분석하여 검색용 키워드 자동 생성"""
    response = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": IMAGE_ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": IMAGE_ANALYSIS_USER_PROMPT_SEARCH},
                {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}}
            ]}
        ],
    )
    result_text = response.choices[0].message.content.strip()
    description, keywords = "", ""
    for line in result_text.split("\n"):
        line = line.strip()
        if line.startswith("설명:"): description = line.replace("설명:", "").strip()
        elif line.startswith("키워드:"): keywords = line.replace("키워드:", "").strip()
    if not description: description = result_text
    if not keywords: keywords = description
    return {"description": description, "keywords": keywords}


def print_results_with_evidence(results, title="검색 결과"):
    """검색 결과 + structured_features 기반 매칭 근거 출력"""
    print(f"\n{'='*85}")
    print(f"{title}")
    print('='*85)
    result_list = list(results)
    if not result_list:
        print("검색 결과가 없습니다.")
        return result_list
    for idx, r in enumerate(result_list, 1):
        score = r.get('@search.score', 0)
        print(f"\n{idx}. [{score:.4f}] {r.get('title', 'N/A')}")
        print(f"   브랜드: {r.get('brand', 'N/A')} | 카테고리: {r.get('category', 'N/A')} | 가격: {r.get('normal_price', 0):,}원")
        sf = r.get('structured_features')
        if sf:
            parts = []
            if sf.get('product_type'): parts.append(f"유형={sf['product_type']}")
            if sf.get('color'): parts.append(f"색상={sf['color']}")
            if sf.get('material'): parts.append(f"소재={sf['material']}")
            if sf.get('style'): parts.append(f"스타일={sf['style']}")
            if sf.get('use_case'): parts.append(f"용도={sf['use_case']}")
            if sf.get('target_customer'): parts.append(f"타겟={sf['target_customer']}")
            print(f"   📋 근거: {' | '.join(parts)}")
        img = r.get('image_link', '')
        if img:
            print(f"   🖼️ {img}")
    return result_list


print("✅ 함수 정의 완료")

✅ 함수 정의 완료


---
## 3️⃣ 모드 1: 이미지 기반 검색 (3가지 서브모드)

하나의 함수에서 `auto_analyze`와 `text` 파라미터로 3가지 실험을 합니다.

| 서브모드 | text | auto_analyze | 동작 |
|---------|------|:---:|------|
| **A. 순수 이미지** | None | False | 이미지→1024D → `image_vector` only |
| **B. 이미지 + 사용자 텍스트** | "운동화" | False | 이미지→1024D + 텍스트→1024D + BM25 |
| **C. 이미지 + GPT 자동분석** | None | **True** | GPT Vision이 텍스트 생성 → B와 동일 경로 |

```
📸 이미지 URL
    ↓
    ├─ [A] auto_analyze=False, text=None
    │      → vectorizeImage → image_vector 검색 (순수 시각)
    │
    ├─ [B] text="사용자 입력"
    │      → vectorizeImage → image_vector
    │      → vectorizeText(사용자텍스트) → mm_vision_text_vector
    │      → BM25(사용자텍스트)
    │      → RRF 통합
    │
    └─ [C] auto_analyze=True
           → GPT Vision → 자동 텍스트 생성
           → vectorizeImage → image_vector
           → vectorizeText(자동텍스트) → mm_vision_text_vector
           → BM25(자동텍스트)
           → RRF 통합
```

> 💡 **실험 목표**: 같은 이미지에 A/B/C를 돌려보면 AI 자동분석이 검색 품질을 올리는지, 사람 의도가 더 나은지 비교 가능

In [10]:
def search_by_image(image_url, text=None, auto_analyze=False, top_k=5):
    """
    이미지 기반 검색 — 3가지 서브모드
    A) text=None, auto_analyze=False → 순수 이미지 벡터 검색
    B) text="...", auto_analyze=False → 이미지 + 사용자 텍스트 하이브리드
    C) text=None, auto_analyze=True  → GPT Vision 자동분석 → 하이브리드
    """
    print("📸 이미지 벡터화 중 (AI Vision → 1024D)...")
    image_vec = vectorize_image(image_url)
    
    if image_vec is None:
        print("❌ 이미지 벡터화 실패. 해당 이미지 URL에 접근할 수 없습니다.")
        return []
    
    search_text = None
    extra_vec = None
    
    if text:
        mode_label = "B. 이미지 + 사용자 텍스트"
        search_text = text
        print(f"💬 사용자 입력: {text}")
        print(f"   텍스트 벡터화 중 (AI Vision → 1024D)...")
        extra_vec = vectorize_text_vision(text)
    elif auto_analyze:
        mode_label = "C. 이미지 + GPT 자동분석"
        print("🤖 GPT Vision 자동 분석 중...")
        analysis = auto_analyze_image(image_url)
        search_text = analysis['keywords']
        print(f"   📝 자동 설명: {analysis['description']}")
        print(f"   🏷️ 자동 키워드: {analysis['keywords']}")
        print(f"   텍스트 벡터화 중 (AI Vision → 1024D)...")
        extra_vec = vectorize_text_vision(analysis['description'])
    else:
        mode_label = "A. 순수 이미지"
    
    vector_queries = [
        VectorizedQuery(vector=image_vec, k_nearest_neighbors=top_k, fields="image_vector", weight=1.5 if extra_vec else 1.0)
    ]
    if extra_vec:
        vector_queries.append(
            VectorizedQuery(vector=extra_vec, k_nearest_neighbors=top_k, fields="mm_vision_text_vector", weight=1.0)
        )
    
    print(f"\n🔀 [{mode_label}] 검색 실행...")
    results = search_client.search(
        search_text=search_text,
        vector_queries=vector_queries,
        select=["title", "brand", "category", "normal_price", "image_link", "structured_features"],
        top=top_k
    )
    return print_results_with_evidence(results, f"🖼️ [{mode_label}] 검색 결과")

---
## 4️⃣ 모드 2: 텍스트만 검색

```
💬 텍스트
    ↓
    ├─ AI Vision vectorizeText → 1024D → mm_vision_text_vector (시각 크로스모달)
    ├─ OpenAI embedding       → 3072D → content_vector (텍스트 의미)
    └─ BM25 키워드 검색
    ↓
    RRF 통합
```

> 💡 **듀얼 벡터 전략**: 같은 텍스트를 Florence(1024D)과 OpenAI(3072D) 두 모델로 벡터화하여 시각+의미 동시 검색

In [11]:
def search_text_only(text, top_k=5):
    """
    모드 2: 텍스트만 → 듀얼 벡터 하이브리드 검색
    
    - 텍스트 → vectorizeText → 1024D → mm_vision_text_vector (시각 크로스모달)
    - 텍스트 → embedding     → 3072D → content_vector (텍스트 의미)
    - 텍스트 → BM25 키워드 검색
    - RRF로 3개 점수 통합
    """
    print(f"💬 입력: {text}")
    
    print("📊 벡터화 1: AI Vision → 1024D (시각 크로스모달 공간)...")
    text_vision_vec = vectorize_text_vision(text)
    
    print("📊 벡터화 2: text-embedding-3-large → 3072D (텍스트 의미 공간)...")
    text_openai_vec = get_embedding(text)
    
    print(f"🔀 듀얼벡터 하이브리드 검색: mm_vision_text_vector + content_vector + BM25")
    
    results = search_client.search(
        search_text=text,
        vector_queries=[
            VectorizedQuery(vector=text_vision_vec, k_nearest_neighbors=top_k, fields="mm_vision_text_vector", weight=1.0),
            VectorizedQuery(vector=text_openai_vec, k_nearest_neighbors=top_k, fields="content_vector", weight=1.0),
        ],
        select=["title", "brand", "category", "normal_price", "image_link", "structured_features"],
        top=top_k
    )
    
    return print_results_with_evidence(results, "💬 모드2: 텍스트만 (mm_vision_text_vector + content_vector + BM25)")


# ── 데모 ──
results_text = search_text_only("빨간색 운동화")

💬 입력: 빨간색 운동화
📊 벡터화 1: AI Vision → 1024D (시각 크로스모달 공간)...
📊 벡터화 2: text-embedding-3-large → 3072D (텍스트 의미 공간)...
🔀 듀얼벡터 하이브리드 검색: mm_vision_text_vector + content_vector + BM25

💬 모드2: 텍스트만 (mm_vision_text_vector + content_vector + BM25)

1. [0.0476] 컨버스 척테일러 올스타 무브 화이트 570257C
   브랜드: 컨버스 | 카테고리: 스포츠/레져 | 가격: 57,800원
   📋 근거: 유형=스니커즈(로우탑) | 색상=['화이트', '오프화이트(솔/미드솔)', '검정(라벨 포인트)'] | 소재=['캔버스(어퍼)', '러버(토캡/아웃솔)', '메탈(아일렛)'] | 스타일=['캐주얼', '스포츠/레저', '미니멀', '플랫폼'] | 용도=['일상', '외출', '가벼운 운동', '여행'] | 타겟=성인(남녀공용)
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/60A1738899_0_600.jpg

2. [0.0320] 컨버스 척테일러 올스타 클래식 블랙 M9166C
   브랜드: 컨버스 | 카테고리: 스포츠/레져 | 가격: 46,920원
   📋 근거: 유형=스니커즈(로우탑 캔버스화) | 색상=['블랙', '화이트'] | 소재=['캔버스', '고무(러버)'] | 스타일=['클래식', '캐주얼', '스포티'] | 용도=['일상', '외출', '가벼운 스포츠/레저'] | 타겟=성인(남녀공용)
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/60A1729602_0_600.jpg

3. [0.0304] [아디다스코리아공식] JS4957 아디제로 보스턴 13 W
   브랜드: 아디다스 | 카테고리: 스포츠/레져 | 가격: 179,000원
   📋 근거

---
## 5️⃣ 통합 멀티모달 검색 함수

입력에 따라 자동으로 최적 검색 전략을 선택합니다.

| 입력 | 함수 | 서브모드 |
|------|------|---------|
| 이미지만 | `search_by_image(auto_analyze=False)` | A. 순수 이미지 |
| 이미지만 + auto | `search_by_image(auto_analyze=True)` | C. GPT 자동분석 |
| 이미지 + 텍스트 | `search_by_image(text=...)` | B. 사용자 텍스트 |
| 텍스트만 | `search_text_only(text)` | 듀얼 벡터 하이브리드 |

In [12]:
def multimodal_search_v2(text=None, image=None, auto_analyze=False, top_k=5):
    """
    멀티모달 통합 검색 v2
    
    - 이미지만 (auto_analyze=False) → 순수 이미지 벡터 검색
    - 이미지만 (auto_analyze=True)  → GPT Vision 자동분석 + 하이브리드
    - 이미지 + 텍스트              → 사용자 텍스트 + 이미지 하이브리드
    - 텍스트만                     → 듀얼 벡터 하이브리드
    """
    if not text and not image:
        print("❌ 텍스트 또는 이미지 중 하나는 입력해야 합니다.")
        return []
    
    if image:
        return search_by_image(image, text=text, auto_analyze=auto_analyze, top_k=top_k)
    else:
        return search_text_only(text, top_k=top_k)


print("✅ multimodal_search_v2 정의 완료")

✅ multimodal_search_v2 정의 완료


---
## 6️⃣ 통합 데모

### 데모 1: 🖼️ 이미지만 A/B/C 비교

같은 이미지에 대해 3가지 서브모드의 결과 차이를 비교합니다.

In [13]:
image_url = "https://image.msscdn.net/thumbnails/mfile_s01/cms-files/67690b26ecdaa3.08995619.jpg"
print("📸 입력 이미지 (뉴발란스 운동화):")
display(IPImage(url=image_url, width=300))

print("\n" + "━"*85)
print("🅰️ 순수 이미지만 (GPT 호출 없음, 가장 빠름)")
print("━"*85)
multimodal_search_v2(image=image_url, top_k=3)

print("\n\n" + "━"*85)
print("🅱️ 이미지 + 사용자 텍스트 (사용자가 의도를 명시)")
print("━"*85)
multimodal_search_v2(image=image_url, text="이 브랜드 운동화를 찾고 있어", top_k=3)

print("\n\n" + "━"*85)
print("🅲 이미지 + GPT 자동분석 (AI가 이미지를 텍스트로 변환)")
print("━"*85)
multimodal_search_v2(image=image_url, auto_analyze=True, top_k=3)

📸 입력 이미지 (뉴발란스 운동화):



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🅰️ 순수 이미지만 (GPT 호출 없음, 가장 빠름)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📸 이미지 벡터화 중 (AI Vision → 1024D)...

🔀 [A. 순수 이미지] 검색 실행...

🖼️ [A. 순수 이미지] 검색 결과

1. [0.7857] [디스커버리] 글라이드 비브람 DXSH4025N (DXSH4025N)
   브랜드: 디스커버리 | 카테고리: 스포츠/레져 | 가격: 219,000원
   📋 근거: 유형=트레일 러닝화/아웃도어 스니커즈 | 색상=['라이트블루/스카이블루', '퍼플', '차콜/블랙'] | 소재=['메쉬', '합성가죽/합성소재 오버레이', '러버(비브람 아웃솔)'] | 스타일=['스포티', '아웃도어', '테크웨어', '유틸리티'] | 용도=['트레킹/하이킹', '트레일 러닝', '캠핑', '일상 워킹'] | 타겟=성인 남녀
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/2238291355_0_600.jpg

2. [0.7839] 컨버스 척테일러 1970s 클래식 162058C
   브랜드: 컨버스 | 카테고리: 스포츠/레져 | 가격: 64,600원
   📋 근거: 유형=스니커즈(캔버스화/로우탑) | 색상=['블랙', '오프화이트(화이트)'] | 소재=['캔버스', '고무(러버)'] | 스타일=['클래식', '캐주얼', '레트로', '미니멀'] | 용도=['일상', '외출', '스포츠/레저', '워크/산책'] | 타겟=성인(남녀공용)
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/60A1939506_0_600.jpg

3. [0.

[{'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/60A1729602_0_600.jpg',
  'title': '컨버스 척테일러 올스타 클래식 블랙 M9166C',
  'structured_features': {'brand': '컨버스(Converse)',
   'model_gender': '유니섹스',
   'product_type': '스니커즈(로우탑 캔버스화)',
   'color': ['블랙', '화이트'],
   'material': ['캔버스', '고무(러버)'],
   'style': ['클래식', '캐주얼', '스포티'],
   'use_case': ['일상', '외출', '가벼운 스포츠/레저'],
   'target_customer': '성인(남녀공용)',
   'feature': ['로우탑 실루엣',
    '레이스업(화이트 슈레이스)',
    '화이트 러버 토캡',
    '화이트 러버 미드솔/아웃솔',
    '측면 대비 스티치(화이트)',
    '금속 아이렛(슈레이스 구멍)',
    '텍스처드(다이아몬드 패턴) 토범퍼',
    '클래식 올스타 디자인',
    '가벼운 착화감(캔버스 어퍼)',
    '고무 아웃솔로 접지력',
    '기본 컬러 조합(블랙/화이트)로 코디 범용성']},
  'brand': '컨버스',
  'normal_price': 46920,
  'category': '스포츠/레져',
  '@search.score': 0.05725365877151489,
  '@search.reranker_score': None,
  '@search.highlights': None,
  '@search.captions': None},
 {'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/2238291355_0_600.jpg',
  'title': '[디스커버리] 글라이

### 데모 2: 💬 텍스트만 검색

In [14]:
multimodal_search_v2(text="출산 선물 세트 추천해줘")

💬 입력: 출산 선물 세트 추천해줘
📊 벡터화 1: AI Vision → 1024D (시각 크로스모달 공간)...
📊 벡터화 2: text-embedding-3-large → 3072D (텍스트 의미 공간)...
🔀 듀얼벡터 하이브리드 검색: mm_vision_text_vector + content_vector + BM25

💬 모드2: 텍스트만 (mm_vision_text_vector + content_vector + BM25)

1. [0.0495] (에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)
   브랜드: 에뜨와 | 카테고리: 유아동 | 가격: 38,000원
   📋 근거: 유형=딸랑이/유아완구 세트 | 색상=['화이트', '파스텔 옐로우', '파스텔 핑크', '파스텔 블루', '민트', '투명'] | 소재=['플라스틱', '패브릭(인형류)'] | 스타일=['귀여운', '파스텔', '선물용', '베이비'] | 용도=['임신/출산 선물', '백일선물', '돌선물', '조카선물', '감각놀이', '일상'] | 타겟=신생아/영유아
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/2142411125_0_600.jpg

2. [0.0484] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동 | 가격: 40,000원
   📋 근거: 유형=딸랑이 세트 | 색상=['오프화이트', '파스텔블루', '파스텔핑크', '피치', '멀티컬러'] | 소재=['패브릭(봉제인형)', '플라스틱(딸랑이/링)', '금속(클립/링)'] | 스타일=['파스텔', '귀여운', '클래식/베이비'] | 용도=['출산선물', '신생아 선물', '집콕/놀이', '외출(유모차/카시트 장착)'] | 타겟=신생아
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/image

[{'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/2142411125_0_600.jpg',
  'title': '(에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종) (유아/출산/백일/돌선물/조카선물)',
  'structured_features': {'brand': '에뜨와(ETTOI)',
   'model_gender': '공용',
   'product_type': '딸랑이/유아완구 세트',
   'color': ['화이트', '파스텔 옐로우', '파스텔 핑크', '파스텔 블루', '민트', '투명'],
   'material': ['플라스틱', '패브릭(인형류)'],
   'style': ['귀여운', '파스텔', '선물용', '베이비'],
   'use_case': ['임신/출산 선물', '백일선물', '돌선물', '조카선물', '감각놀이', '일상'],
   'target_customer': '신생아/영유아',
   'feature': ['6종 구성 세트',
    '선물 패키지 박스(리본/브랜딩)',
    '딸랑이/흔들면 소리 나는 완구',
    '투명 링 내부 컬러 비즈(시각 자극)',
    '하트 모티프',
    '테디베어 인형 포함',
    '파스텔 톤 컬러로 부드러운 인상',
    '가벼운 링/그립 형태로 아기 손에 쥐기 쉬움']},
  'brand': '에뜨와',
  'normal_price': 38000,
  'category': '유아동',
  '@search.score': 0.049453552812337875,
  '@search.reranker_score': None,
  '@search.highlights': None,
  '@search.captions': None},
 {'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/2130369107_0

---
### 데모 3: 🧪 다른 이미지로 A/B/C 비교 (유아동 카테고리)

같은 실험을 유아동 제품 이미지로도 해봅니다. 카테고리가 다를 때 auto_analyze의 효과가 어떤지 비교합니다.

In [15]:
# 블루독베이비 손수건세트 (id=3, 유아동)
baby_url = "https://foundryiq-image-gallery.azurewebsites.net/images/2138687276_0_600.jpg"
print("📸 입력 이미지 (블루독베이비 손수건세트):")
display(IPImage(url=baby_url, width=300))

print("\n" + "━"*85)
print("🅰️ 순수 이미지만")
print("━"*85)
multimodal_search_v2(image=baby_url, top_k=3)

print("\n\n" + "━"*85)
print("🅱️ 이미지 + 사용자 텍스트")
print("━"*85)
multimodal_search_v2(image=baby_url, text="신생아 손수건 선물", top_k=3)

print("\n\n" + "━"*85)
print("🅲 이미지 + GPT 자동분석")
print("━"*85)
multimodal_search_v2(image=baby_url, auto_analyze=True, top_k=3)

📸 입력 이미지 (블루독베이비 손수건세트):



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🅰️ 순수 이미지만
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📸 이미지 벡터화 중 (AI Vision → 1024D)...

🔀 [A. 순수 이미지] 검색 실행...

🖼️ [A. 순수 이미지] 검색 결과

1. [1.0000] 블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손수건/타올/수건/10장세트/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동 | 가격: 22,190원
   📋 근거: 유형=손수건 세트 | 색상=['화이트', '민트', '블루', '옐로우', '네이비'] | 소재=['면(추정)'] | 스타일=['베이비', '미니멀', '귀여운', '캐주얼'] | 용도=['출산선물', '신생아용', '수유/침흘림 케어', '일상'] | 타겟=신생아
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/2138687276_0_600.jpg

2. [0.9145] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   브랜드: 블루독베이비 | 카테고리: 유아동 | 가격: 22,190원
   📋 근거: 유형=손수건 세트 | 색상=['화이트', '파스텔 핑크', '파스텔 퍼플', '파스텔 민트'] | 소재=['뱀부(밤부)', '면(추정)'] | 스타일=['귀여운', '미니멀', '베이비'] | 용도=['출산선물', '신생아용', '수유', '침/땀 닦기', '일상'] | 타겟=신생아
   🖼️ https://foundryiq-image-gallery.azurewebsites.net/images/2138687338_0_600.jpg

[{'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/2138687276_0_600.jpg',
  'title': '블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손수건/타올/수건/10장세트/신생아/출산용품/선물',
  'structured_features': {'brand': '블루독베이비(BLUEDOG baby)',
   'model_gender': '남아',
   'product_type': '손수건 세트',
   'color': ['화이트', '민트', '블루', '옐로우', '네이비'],
   'material': ['면(추정)'],
   'style': ['베이비', '미니멀', '귀여운', '캐주얼'],
   'use_case': ['출산선물', '신생아용', '수유/침흘림 케어', '일상'],
   'target_customer': '신생아',
   'feature': ['10장 세트',
    '남아용 컬러/패턴 구성',
    '강아지(블루독) 캐릭터/로고 프린트',
    '올오버 패턴(발자국/도트류) 디자인',
    '선물용 박스 패키징(윈도우 박스)',
    '휴대/교체가 쉬운 다수 구성']},
  'brand': '블루독베이비',
  'normal_price': 22190,
  'category': '유아동',
  '@search.score': 0.057522475719451904,
  '@search.reranker_score': None,
  '@search.highlights': None,
  '@search.captions': None},
 {'image_link': 'https://foundryiq-image-gallery.azurewebsites.net/images/2138687338_0_600.jpg',
  'title': '블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10

---
## 📊 정리

### 실험 설계: 이미지 검색 3가지 서브모드

| 서브모드 | GPT Vision 호출 | 검색 경로 | 비용 | 기대 효과 |
|---------|:---:|------|------|---------|
| **A. 순수 이미지** | ✗ | image_vector only | 가장 저렴 | 시각적으로 비슷한 것만 |
| **B. + 사용자 텍스트** | ✗ | image_vector + mm_vision_text_vector + BM25 | 저렴 | 사용자 의도 반영 |
| **C. + GPT 자동분석** | ✓ | image_vector + mm_vision_text_vector + BM25 | GPT 호출 1회 | AI가 의도 추론 |

### 실험으로 확인할 수 있는 것

1. **A vs C**: 순수 이미지만으로 충분한가? → GPT 자동분석이 검색 정확도를 올리는가?
2. **B vs C**: 사람이 직접 의도를 쓰는 게 나은가? AI 자동분석이 나은가?
3. **A vs B**: 사용자 텍스트가 검색 정확도에 얼마나 기여하는가?

### 전체 검색 전략

| 모드 | 쿼리 벡터 | 검색 필드 | BM25 |
|------|----------|----------|:---:|
| **이미지 (A)** | 이미지→1024D | `image_vector` | ✗ |
| **이미지 (B)** | 이미지→1024D + 텍스트→1024D | `image_vector` + `mm_vision_text_vector` | ✓ |
| **이미지 (C)** | 이미지→1024D + GPT텍스트→1024D | `image_vector` + `mm_vision_text_vector` | ✓ |
| **텍스트만** | 텍스트→1024D + 텍스트→3072D | `mm_vision_text_vector` + `content_vector` | ✓ |

### 핵심 설계 원칙

1. **같은 공간 활용**: `image_vector`와 `mm_vision_text_vector`는 동일 Florence 1024D 공간
2. **듀얼 벡터**: 텍스트만 입력 시 Vision(1024D)과 OpenAI(3072D) 동시 검색
3. **근거 제공**: 모든 결과에 `structured_features`로 매칭 이유 표시
4. **실험 가능**: `auto_analyze` 토글로 AI 자동분석 효과를 A/B 테스트

---
## 8️⃣ Florence 1024D 공간 검증: 이미지↔텍스트 코사인 유사도

Azure AI Vision Florence 모델은 CLIP과 유사한 **contrastive learning** 아키텍처입니다.  
이미지와 텍스트가 정말 같은 공간에 있는지, 코사인 유사도로 직접 검증합니다.

### 실험 설계

| 비교 | 기대 유사도 |
|------|-----------|
| 운동화 이미지 ↔ "검정 운동화" 텍스트 | **높음** (같은 제품) |
| 운동화 이미지 ↔ "아기 손수건 세트" 텍스트 | **낮음** (다른 카테고리) |
| 손수건 이미지 ↔ "아기 손수건 세트" 텍스트 | **높음** (같은 제품) |
| 손수건 이미지 ↔ "검정 운동화" 텍스트 | **낮음** (다른 카테고리) |

In [18]:
import numpy as np
import pandas as pd

def cosine_similarity(a, b):
    """두 벡터의 코사인 유사도"""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# ── 인덱스에 저장된 제품들로 검증 ──
# 다양한 카테고리에서 샘플 선택
df = pd.read_csv("../00-data/sample_data.csv")
samples = [
    df[df['category'] == '스포츠/레져'].iloc[0],   # 운동화/레깅스
    df[df['category'] == '스포츠/레져'].iloc[1],
    df[df['category'] == '유아동'].iloc[0],         # 손수건/딸랑이
    df[df['category'] == '유아동'].iloc[1],
    df[df['category'] == '패션의류'].iloc[0],       # 의류
    df[df['category'] == '패션잡화'].iloc[0],       # 가방/신발
]

print("📊 인덱스 제품별 image_vector ↔ mm_vision_text_vector 정렬 품질 측정")
print("   (같은 제품의 이미지와 GPT Vision 분석 텍스트가 Florence 공간에서 얼마나 가까운지)")
print('='*90)

results = []

for row in samples:
    title = row['title'][:40]
    img_url = row['image_link']
    category = row['category']
    
    # 이 제품의 enriched_content 텍스트를 간단히 재구성 (인덱싱 시와 유사)
    text_desc = f"{row['title']} {row['brand']} {row['category']}"
    
    print(f"\n🔍 [{category}] {title}...")
    
    # 1) 이미지 → vectorizeImage → 1024D
    img_vec = vectorize_image(img_url)
    if img_vec is None:
        print("   ⚠️ 이미지 벡터화 실패, 건너뜀")
        continue
    
    # 2) 제품 텍스트 → vectorizeText → 1024D (mm_vision_text_vector와 같은 경로)
    text_vec = vectorize_text_vision(text_desc)
    
    # 3) 코사인 유사도
    sim = cosine_similarity(img_vec, text_vec)
    results.append({"title": title, "category": category, "similarity": sim})
    print(f"   이미지 ↔ 텍스트 유사도: {sim:.4f}")

# ── 크로스 비교: 다른 제품끼리 ──
print(f"\n\n{'='*90}")
print("📊 같은 제품 vs 다른 카테고리 제품 간 유사도 비교")
print('='*90)

# 운동화 이미지 ↔ 유아동 텍스트 (다른 카테고리)
sport_img = vectorize_image(df[df['category'] == '스포츠/레져'].iloc[0]['image_link'])
baby_row = df[df['category'] == '유아동'].iloc[0]
baby_text = vectorize_text_vision(f"{baby_row['title']} {baby_row['brand']} {baby_row['category']}")
cross_sim = cosine_similarity(sport_img, baby_text)

print(f"\n{'비교':<50} {'유사도':<10}")
print("─"*60)

avg_same = np.mean([r['similarity'] for r in results])
print(f"{'같은 제품 (이미지 ↔ 자기 텍스트) 평균':<50} {avg_same:.4f} ✅")
print(f"{'다른 카테고리 (운동화 이미지 ↔ 유아동 텍스트)':<50} {cross_sim:.4f}")
print(f"{'차이 (Δ)':<50} {avg_same - cross_sim:+.4f}")

print(f"\n{'='*90}")
print("💡 결론")
print('='*90)
print(f"• 같은 제품의 이미지↔텍스트 평균 유사도: {avg_same:.4f}")
print(f"• 다른 카테고리 간 유사도: {cross_sim:.4f}")
print(f"• 차이(Δ): {avg_same - cross_sim:+.4f}")
print(f"\n→ Florence 모델이 같은 제품의 이미지와 텍스트를 동일 공간에서")
print(f"  일관되게 정렬하고 있으며, image_vector로 검색 → mm_vision_text_vector로")
print(f"  크로스모달 매칭이 유효함을 확인")

# 제품별 상세
print(f"\n{'─'*90}")
print(f"{'제품':<45} {'카테고리':<12} {'유사도'}")
print(f"{'─'*90}")
for r in sorted(results, key=lambda x: x['similarity'], reverse=True):
    bar = "█" * int(r['similarity'] * 40)
    print(f"{r['title']:<45} {r['category']:<12} {r['similarity']:.4f} {bar}")

📊 인덱스 제품별 image_vector ↔ mm_vision_text_vector 정렬 품질 측정
   (같은 제품의 이미지와 GPT Vision 분석 텍스트가 Florence 공간에서 얼마나 가까운지)

🔍 [스포츠/레져] [젝시믹스] V업 3D 플러스 레깅스 1+1 (XP9156T)...
   이미지 ↔ 텍스트 유사도: 0.4065

🔍 [스포츠/레져] [젝시믹스] 아이스페더 컴포트 숏슬리브 1+1 (XA5298T)...
   이미지 ↔ 텍스트 유사도: 0.4300

🔍 [유아동] 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   이미지 ↔ 텍스트 유사도: 0.3798

🔍 [유아동] [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)...
   이미지 ↔ 텍스트 유사도: 0.3919

🔍 [패션의류] 앤더슨벨 커트 스트라이프 롱슬리브 티셔츠 atb1247m(ECRU)...
   이미지 ↔ 텍스트 유사도: 0.3456

🔍 [패션잡화] 앤더슨벨 aaa425u 유니섹스 테크니컬 베를린 스몰 백팩 가방 블랙...
   이미지 ↔ 텍스트 유사도: 0.3740


📊 같은 제품 vs 다른 카테고리 제품 간 유사도 비교

비교                                                 유사도       
────────────────────────────────────────────────────────────
같은 제품 (이미지 ↔ 자기 텍스트) 평균                            0.3880 ✅
다른 카테고리 (운동화 이미지 ↔ 유아동 텍스트)                        0.2418
차이 (Δ)                                             +0.1461

💡 결론
• 같은 제품의 이미지↔텍스트 평균 유사도: 0.3880
• 다른 카테고리 간 유사도: 0.2418
• 차이(Δ): +0.1461

→ Florence 모델이 같은 

---
## 7️⃣ 프리미엄 모드: 2-Stage 검색

A 결과를 **즉시** 반환하고, 백그라운드로 GPT 자동분석 후 C 결과로 **리랭킹**합니다.

```
📸 이미지
    ↓
[Stage 1] vectorizeImage → image_vector 검색 → 🔹 즉시 결과 반환 (A)
    ↓ (동시에)
[Stage 2] GPT Vision → 자동 텍스트 생성 → image_vector + mm_vision_text_vector + BM25
    ↓
🔹 리랭킹 결과 반환 (C) — A 결과와 비교하여 순위 변화 표시
```

> 💡 **실서비스 패턴**: 사용자에게 A 결과를 0.5초 만에 보여주고, 2~3초 후 "더 정확한 결과"로 업데이트

In [16]:
import time

def search_2stage(image_url, top_k=5):
    """
    2-Stage 프리미엄 검색
    
    Stage 1: 순수 이미지(A) → 즉시 결과
    Stage 2: GPT 자동분석(C) → 리랭킹 결과 + A와 순위 변화 비교
    """
    # ═══ Stage 1: 즉시 결과 (A) ═══
    t0 = time.time()
    print("⚡ [Stage 1] 순수 이미지 벡터 검색 (즉시 응답)...")
    image_vec = vectorize_image(image_url)
    if image_vec is None:
        print("❌ 이미지 벡터화 실패")
        return [], []
    
    results_a = search_client.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=image_vec, k_nearest_neighbors=top_k, fields="image_vector")],
        select=["id", "title", "brand", "category", "normal_price", "image_link", "structured_features"],
        top=top_k
    )
    stage1_results = list(results_a)
    stage1_time = time.time() - t0
    
    print(f"\n🔹 Stage 1 완료 ({stage1_time:.1f}초)")
    stage1_ids = [r['id'] for r in stage1_results]
    for idx, r in enumerate(stage1_results, 1):
        print(f"   {idx}. [{r.get('@search.score', 0):.4f}] {r.get('title', 'N/A')[:50]}")
    
    # ═══ Stage 2: GPT 자동분석 → 리랭킹 (C) ═══
    t1 = time.time()
    print(f"\n🤖 [Stage 2] GPT Vision 자동분석 + 리랭킹 중...")
    analysis = auto_analyze_image(image_url)
    print(f"   📝 자동 설명: {analysis['description'][:80]}...")
    print(f"   🏷️ 자동 키워드: {analysis['keywords'][:80]}...")
    
    text_vision_vec = vectorize_text_vision(analysis['description'])
    
    results_c = search_client.search(
        search_text=analysis['keywords'],
        vector_queries=[
            VectorizedQuery(vector=image_vec, k_nearest_neighbors=top_k, fields="image_vector", weight=1.5),
            VectorizedQuery(vector=text_vision_vec, k_nearest_neighbors=top_k, fields="mm_vision_text_vector", weight=1.0),
        ],
        select=["id", "title", "brand", "category", "normal_price", "image_link", "structured_features"],
        top=top_k
    )
    stage2_results = list(results_c)
    stage2_time = time.time() - t1
    stage2_ids = [r['id'] for r in stage2_results]
    
    print(f"\n🔹 Stage 2 완료 ({stage2_time:.1f}초)")
    
    # ═══ 순위 변화 비교 ═══
    print(f"\n{'='*85}")
    print(f"📊 2-Stage 리랭킹 결과 비교")
    print(f"   Stage 1 (순수 이미지): {stage1_time:.1f}초 | Stage 2 (GPT 분석): {stage2_time:.1f}초")
    print('='*85)
    
    for idx, r in enumerate(stage2_results, 1):
        title = r.get('title', 'N/A')[:50]
        score = r.get('@search.score', 0)
        rid = r['id']
        
        # Stage 1에서의 순위 찾기
        if rid in stage1_ids:
            old_rank = stage1_ids.index(rid) + 1
            if old_rank < idx:
                change = f"🔻 {old_rank}→{idx}"
            elif old_rank > idx:
                change = f"🔺 {old_rank}→{idx}"
            else:
                change = f"━━ {idx}위 유지"
        else:
            change = f"🆕 신규 진입"
        
        print(f"\n{idx}. [{score:.4f}] {title}")
        print(f"   {change}")
        
        sf = r.get('structured_features')
        if sf:
            parts = []
            if sf.get('product_type'): parts.append(f"유형={sf['product_type']}")
            if sf.get('color'): parts.append(f"색상={sf['color']}")
            if sf.get('style'): parts.append(f"스타일={sf['style']}")
            print(f"   📋 근거: {' | '.join(parts)}")
    
    # 변화 요약
    new_entries = [r['id'] for r in stage2_results if r['id'] not in stage1_ids]
    dropped = [rid for rid in stage1_ids if rid not in stage2_ids]
    
    print(f"\n{'─'*85}")
    print(f"📈 변화 요약: 신규 진입 {len(new_entries)}개 | 탈락 {len(dropped)}개 | 유지 {top_k - len(new_entries)}개")
    if new_entries:
        for nid in new_entries:
            nr = next(r for r in stage2_results if r['id'] == nid)
            print(f"   🆕 {nr.get('title', 'N/A')[:50]}")
    if dropped:
        for did in dropped:
            dr = next(r for r in stage1_results if r['id'] == did)
            print(f"   ❌ {dr.get('title', 'N/A')[:50]}")
    
    return stage1_results, stage2_results


# ── 데모 ──
print("📸 입력 이미지 (뉴발란스 운동화):")
image_url = "https://image.msscdn.net/thumbnails/mfile_s01/cms-files/67690b26ecdaa3.08995619.jpg"
display(IPImage(url=image_url, width=300))
stage1, stage2 = search_2stage(image_url, top_k=5)

📸 입력 이미지 (뉴발란스 운동화):


⚡ [Stage 1] 순수 이미지 벡터 검색 (즉시 응답)...

🔹 Stage 1 완료 (0.5초)
   1. [0.7857] [디스커버리] 글라이드 비브람 DXSH4025N (DXSH4025N)
   2. [0.7839] 컨버스 척테일러 1970s 클래식 162058C
   3. [0.7756] 컨버스 척테일러 올스타 클래식 블랙 M9166C
   4. [0.7752] [그랜드스테이지] 써코니 엔돌핀 프로 4 M S20939-30
   5. [0.7714] [그랜드스테이지] 써코니 허리케인 24 M S20933-500

🤖 [Stage 2] GPT Vision 자동분석 + 리랭킹 중...
   📝 자동 설명: 올블랙 컬러를 기반으로 한 뉴발란스(N) 로고가 크게 배치된 로우탑 러닝화/라이프스타일 스니커즈입니다. 블랙 바디에 화이트·실버 톤의 로고 테두...
   🏷️ 자동 키워드: 뉴발란스, 1000 스니커즈, 올블랙 운동화, 블랙 러닝화, Y2K 레트로 러너, 메쉬 어퍼, 합성가죽 패널, 남녀공용 유니섹스, 데일리 워킹화...

🔹 Stage 2 완료 (16.9초)

📊 2-Stage 리랭킹 결과 비교
   Stage 1 (순수 이미지): 0.5초 | Stage 2 (GPT 분석): 16.9초

1. [0.0558] 컨버스 척테일러 올스타 클래식 블랙 M9166C
   🔺 3→1
   📋 근거: 유형=스니커즈(로우탑 캔버스화) | 색상=['블랙', '화이트'] | 스타일=['클래식', '캐주얼', '스포티']

2. [0.0551] [디스커버리] 글라이드 비브람 DXSH4025N (DXSH4025N)
   🔻 1→2
   📋 근거: 유형=트레일 러닝화/아웃도어 스니커즈 | 색상=['라이트블루/스카이블루', '퍼플', '차콜/블랙'] | 스타일=['스포티', '아웃도어', '테크웨어', '유틸리티']

3. [0.0550] 컨버스 척테일러 1970s 클래식 162058C
   🔻 2→3
   📋 근거: 유형=스니커즈(캔버스화/로우탑) | 색상=['블랙', 

---
## 📊 정리

### 검색 전략 비교

| | 모드1: 이미지만 | 모드2: 이미지+텍스트 | 모드3: 텍스트만 |
|---|---|---|---|
| **쿼리 벡터** | 이미지→1024D | 이미지→1024D + 텍스트→1024D | 텍스트→1024D + 텍스트→3072D |
| **검색 필드** | `image_vector` | `image_vector` + `mm_vision_text_vector` | `mm_vision_text_vector` + `content_vector` |
| **BM25** | ✗ | ✓ | ✓ |
| **점수 통합** | 벡터만 | RRF (3-way) | RRF (3-way) |
| **매칭 근거** | `structured_features` | `structured_features` | `structured_features` |
| **강점** | 순수 시각 유사도 | 시각 + 의미 결합 | 시각 개념 + 텍스트 의미 동시 |

### 핵심 설계 원칙

1. **같은 공간 활용**: `image_vector`와 `mm_vision_text_vector`는 동일한 Florence 1024D 공간 → 이미지↔텍스트 크로스모달 검색 가능
2. **듀얼 벡터**: 텍스트만 입력 시 Vision 공간(1024D)과 OpenAI 공간(3072D)에서 동시 검색 → 더 넓은 recall
3. **근거 제공**: 모든 결과에 `structured_features`를 포함하여 **왜 이 제품이 매칭되었는지** 설명 가능
4. **멀티벡터 RRF**: Azure AI Search가 여러 벡터 쿼리의 점수를 자동으로 RRF 통합